## Step 1: Load the Dataset and Define Target Variable

First, we load the `nba-players.csv` dataset using pandas. Then, we identify `target_5yrs` as our dependent variable for the machine learning task.

In [10]:
import pandas as pd

# Load the dataset
df = pd.read_csv('/content/nba-players.csv')

# Display the first few rows and check the dataset's information
display(df.head())
df.info()

,Unnamed: 0,name,gp,min,pts,fgm,fga,fg,3p_made,3pa,...,fta,ft,oreb,dreb,reb,ast,stl,blk,tov,target_5yrs
0,0,Brandon Ingram,36,27.4,7.4,2.6,7.6,34.7,0.5,2.1,...,2.3,69.9,0.7,3.4,4.1,1.9,0.4,0.4,1.3,0
1,1,Andrew Harrison,35,26.9,7.2,2.0,6.7,29.6,0.7,2.8,...,3.4,76.5,0.5,2.0,2.4,3.7,1.1,0.5,1.6,0
2,2,JaKarr Sampson,74,15.3,5.2,2.0,4.7,42.2,0.4,1.7,...,1.3,67.0,0.5,1.7,2.2,1.0,0.5,0.3,1.0,0
3,3,Malik Sealy,58,11.6,5.7,2.3,5.5,42.6,0.1,0.5,...,1.3,68.9,1.0,0.9,1.9,0.8,0.6,0.1,1.0,1
4,4,Matt Geiger,48,11.5,4.5,1.6,3.0,52.4,0.0,0.1,...,1.9,67.4,1.0,1.5,2.5,0.3,0.3,0.4,0.8,1


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1340 entries, 0 to 1339
Data columns (total 22 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Unnamed: 0   1340 non-null   int64  
 1   name         1340 non-null   object 
 2   gp           1340 non-null   int64  
 3   min          1340 non-null   float64
 4   pts          1340 non-null   float64
 5   fgm          1340 non-null   float64
 6   fga          1340 non-null   float64
 7   fg           1340 non-null   float64
 8   3p_made      1340 non-null   float64
 9   3pa          1340 non-null   float64
 10  3p           1340 non-null   float64
 11  ftm          1340 non-null   float64
 12  fta          1340 non-null   float64
 13  ft           1340 non-null   float64
 14  oreb         1340 non-null   float64
 15  dreb         1340 non-null   float64
 16  reb          1340 non-null   float64
 17  ast          1340 non-null   float64
 18  stl          1340 non-null   float64
 19  blk   

## Step 2: Drop Non-Predictive Columns

Columns like `player` (player name), `draft_round`, and `draft_peak` are non-predictive, as they either serve as identifiers or provide information that might not be available at the time of prediction, or introduce data leakage. We'll also drop `target_5yrs` from the features dataframe to prevent data leakage and define it as the dependent variable (target).

In [5]:
# Define the target variable
target = df['target_5yrs']

# Drop non-predictive columns like player name and ID
columns_to_drop = ['Unnamed: 0', 'name', 'target_5yrs']

# Check if columns exist before dropping
existing_columns_to_drop = [col for col in columns_to_drop if col in df.columns]
df_features = df.drop(columns=existing_columns_to_drop)

print(f"Dropped columns: {existing_columns_to_drop}")
display(df_features.head())

Dropped columns: ['Unnamed: 0', 'name', 'target_5yrs']


,gp,min,pts,fgm,fga,fg,3p_made,3pa,3p,ftm,fta,ft,oreb,dreb,reb,ast,stl,blk,tov
0,36,27.4,7.4,2.6,7.6,34.7,0.5,2.1,25.0,1.6,2.3,69.9,0.7,3.4,4.1,1.9,0.4,0.4,1.3
1,35,26.9,7.2,2.0,6.7,29.6,0.7,2.8,23.5,2.6,3.4,76.5,0.5,2.0,2.4,3.7,1.1,0.5,1.6
2,74,15.3,5.2,2.0,4.7,42.2,0.4,1.7,24.4,0.9,1.3,67.0,0.5,1.7,2.2,1.0,0.5,0.3,1.0
3,58,11.6,5.7,2.3,5.5,42.6,0.1,0.5,22.6,0.9,1.3,68.9,1.0,0.9,1.9,0.8,0.6,0.1,1.0
4,48,11.5,4.5,1.6,3.0,52.4,0.0,0.1,0.0,1.3,1.9,67.4,1.0,1.5,2.5,0.3,0.3,0.4,0.8


## Step 3: Perform Correlation Analysis

We will now compute the correlation matrix of our features to identify highly correlated variables. This step helps in reducing redundancy and multicollinearity, which can be problematic for machine learning models. We'll identify pairs of features with a correlation coefficient above a certain threshold (e.g., 0.9).

In [6]:
# Calculate the correlation matrix
correlation_matrix = df_features.corr()

# Display the correlation matrix (first few rows/cols)
display(correlation_matrix.head())

# Identify highly correlated features
high_corr_threshold = 0.9
highly_correlated_pairs = []

# Iterate through the correlation matrix to find highly correlated pairs
# We only need to check the upper triangle to avoid duplicates and self-correlation
for i in range(len(correlation_matrix.columns)):
    for j in range(i + 1, len(correlation_matrix.columns)):
        if abs(correlation_matrix.iloc[i, j]) > high_corr_threshold:
            highly_correlated_pairs.append(
                (correlation_matrix.columns[i],
                 correlation_matrix.columns[j],
                 correlation_matrix.iloc[i, j])
            )

# Print the highly correlated pairs
if highly_correlated_pairs:
    print(f"\nHighly correlated feature pairs (>{high_corr_threshold}):")
    for feature1, feature2, correlation_value in highly_correlated_pairs:
        print(f"- {feature1} and {feature2}: {correlation_value:.2f}")
else:
    print(f"\nNo highly correlated feature pairs found (>{high_corr_threshold}).")


,gp,min,pts,fgm,fga,fg,3p_made,3pa,3p,ftm,fta,ft,oreb,dreb,reb,ast,stl,blk,tov
gp,1.000000,0.590240,0.538471,0.542724,0.516625,0.296289,0.107423,0.098772,0.037133,0.482123,0.479487,0.196299,0.401136,0.466840,0.460406,0.372749,0.451137,0.276498,0.518167
min,0.590240,1.000000,0.911822,0.903060,0.910247,0.203901,0.389920,0.403258,0.168070,0.791000,0.779609,0.239878,0.573062,0.745513,0.709707,0.629015,0.757034,0.399088,0.826500
pts,0.538471,0.911822,1.000000,0.990834,0.979733,0.255333,0.346682,0.356751,0.154955,0.896297,0.880703,0.258931,0.575106,0.693934,0.676849,0.552338,0.675341,0.387043,0.850366
fgm,0.542724,0.903060,0.990834,1.000000,0.980050,0.291693,0.289007,0.299057,0.122542,0.848019,0.840408,0.223566,0.596687,0.703278,0.691186,0.532534,0.662640,0.398125,0.834352
fga,0.516625,0.910247,0.979733,0.980050,1.000000,0.129798,0.390253,0.413560,0.201186,0.826616,0.805559,0.269614,0.504212,0.640123,0.614328,0.589818,0.690168,0.322184,0.845989



Highly correlated feature pairs (>0.9):
- min and pts: 0.91
- min and fgm: 0.90
- min and fga: 0.91
- pts and fgm: 0.99
- pts and fga: 0.98
- fgm and fga: 0.98
- 3p_made and 3pa: 0.98
- ftm and fta: 0.98
- oreb and reb: 0.93
- dreb and reb: 0.98


## Step 4: Engineer New Composite Features

To enrich our dataset and potentially capture more nuanced aspects of player performance, we'll engineer at least two new composite features:

1.  **Points Per Minute (PPM)**: Calculated as `pts / min`. This metric normalizes scoring ability by playing time.
2.  **Efficiency Rating (EFF)**: A common basketball metric that combines various positive and negative contributions. A simplified version can be calculated as `(PTS + REB + AST + STL + BLK - ((FGA - FGM) + (FTA - FTM) + TOV)) / GP`.

In [7]:
# Engineer Points Per Minute (PPM)
# Handle division by zero for 'min' if players have 0 minutes played
df_features['PPM'] = df_features.apply(lambda row: row['pts'] / row['min'] if row['min'] > 0 else 0, axis=1)

# Engineer Efficiency Rating (EFF)
# Using a simplified formula, ensure all columns exist before calculation
required_eff_cols = ['pts', 'reb', 'ast', 'stl', 'blk', 'fga', 'fgm', 'fta', 'ftm', 'tov', 'gp']
if all(col in df_features.columns for col in required_eff_cols):
    df_features['EFF'] = (
        df_features['pts'] +
        df_features['reb'] +
        df_features['ast'] +
        df_features['stl'] +
        df_features['blk'] -
        ((df_features['fga'] - df_features['fgm']) +
         (df_features['fta'] - df_features['ftm']) +
         df_features['tov'])
    ) / df_features['gp']
    # Handle division by zero for 'gp' if players have 0 games played
    df_features['EFF'] = df_features.apply(lambda row: row['EFF'] if row['gp'] > 0 else 0, axis=1)
else:
    print("Warning: Not all columns required for Efficiency Rating are present. EFF will not be created.")

display(df_features[['min', 'pts', 'PPM', 'EFF']].head())







,min,pts,PPM,EFF
0,27.4,7.4,0.270073,0.200000
1,26.9,7.2,0.267658,0.222857
2,15.3,5.2,0.339869,0.068919
3,11.6,5.7,0.491379,0.077586
4,11.5,4.5,0.391304,0.108333


## Step 5: Reduce Redundancy from Highly Correlated Features

Based on the correlation analysis, we identified several highly correlated pairs. To avoid multicollinearity and simplify the model, we will drop one feature from each highly correlated pair. The choice of which feature to drop can sometimes be domain-specific or based on further analysis (e.g., feature importance). For now, we will choose to remove `min`, `fga`, `3pa`, `fta`, and `dreb` as `pts`, `fgm`, `3p_made`, `ftm`, and `reb` are often more direct measures or commonly used.

In [8]:
# Drop selected highly correlated features
features_to_drop_corr = ['min', 'fga', '3pa', 'fta', 'dreb']

# Check if columns exist before dropping
existing_features_to_drop_corr = [col for col in features_to_drop_corr if col in df_features.columns]
df_features = df_features.drop(columns=existing_features_to_drop_corr)

print(f"Dropped highly correlated columns: {existing_features_to_drop_corr}")
display(df_features.head())

Dropped highly correlated columns: ['min', 'fga', '3pa', 'fta', 'dreb']


,gp,pts,fgm,fg,3p_made,3p,ftm,ft,oreb,reb,ast,stl,blk,tov,PPM,EFF
0,36,7.4,2.6,34.7,0.5,25.0,1.6,69.9,0.7,4.1,1.9,0.4,0.4,1.3,0.270073,0.200000
1,35,7.2,2.0,29.6,0.7,23.5,2.6,76.5,0.5,2.4,3.7,1.1,0.5,1.6,0.267658,0.222857
2,74,5.2,2.0,42.2,0.4,24.4,0.9,67.0,0.5,2.2,1.0,0.5,0.3,1.0,0.339869,0.068919
3,58,5.7,2.3,42.6,0.1,22.6,0.9,68.9,1.0,1.9,0.8,0.6,0.1,1.0,0.491379,0.077586
4,48,4.5,1.6,52.4,0.0,0.0,1.3,67.4,1.0,2.5,0.3,0.3,0.4,0.8,0.391304,0.108333


## Step 6: Clean the Dataset by Handling Null Values

Before proceeding with machine learning, it's crucial to handle any missing values in our performance columns. We will first check for null values and then impute them, for example, using the mean or median of each column. This ensures that our dataset is complete and ready for model training.

In [9]:
# Check for null values in the feature DataFrame
print("Null values before imputation:")
display(df_features.isnull().sum()[df_features.isnull().sum() > 0])

# Impute missing values for numerical columns.
# A common strategy is to fill with the mean or median.
# For this dataset, we'll use the mean for simplicity, as most are continuous performance metrics.

for col in df_features.columns:
    if df_features[col].isnull().any():
        if df_features[col].dtype in ['float64', 'int64']:
            df_features[col] = df_features[col].fillna(df_features[col].mean())
            print(f"Filled missing values in column '{col}' with its mean.")
        else:
            # For non-numeric columns, consider other strategies or simply drop
            print(f"Column '{col}' has non-numeric missing values. Not handled by mean imputation.")

print("\nNull values after imputation:")
display(df_features.isnull().sum()[df_features.isnull().sum() > 0])

display(df_features.head())

Null values before imputation:


,0



Null values after imputation:


,0


,gp,pts,fgm,fg,3p_made,3p,ftm,ft,oreb,reb,ast,stl,blk,tov,PPM,EFF
0,36,7.4,2.6,34.7,0.5,25.0,1.6,69.9,0.7,4.1,1.9,0.4,0.4,1.3,0.270073,0.200000
1,35,7.2,2.0,29.6,0.7,23.5,2.6,76.5,0.5,2.4,3.7,1.1,0.5,1.6,0.267658,0.222857
2,74,5.2,2.0,42.2,0.4,24.4,0.9,67.0,0.5,2.2,1.0,0.5,0.3,1.0,0.339869,0.068919
3,58,5.7,2.3,42.6,0.1,22.6,0.9,68.9,1.0,1.9,0.8,0.6,0.1,1.0,0.491379,0.077586
4,48,4.5,1.6,52.4,0.0,0.0,1.3,67.4,1.0,2.5,0.3,0.3,0.4,0.8,0.391304,0.108333


## Step 7: Document Feature Selection and Engineering Choices

This section summarizes the decisions made regarding feature selection and engineering throughout the data preparation process.

### Feature Selection:

1.  **Dropped Non-Predictive Columns**: Initial columns such as `'Unnamed: 0'` (an index column), `'name'` (player names), and `'target_5yrs'` (the dependent variable) were removed from the feature set (`df_features`). These columns are non-predictive or would lead to data leakage.
2.  **Reduced Highly Correlated Features**: Based on the correlation analysis (threshold > 0.9), redundant features were dropped to mitigate multicollinearity and simplify the model. The following features were removed, prioritizing retention of more direct or commonly used metrics:
    *   `min` (correlated with `pts`, `fgm`, `fga`)
    *   `fga` (correlated with `pts`, `fgm`)
    *   `3pa` (correlated with `3p_made`)
    *   `fta` (correlated with `ftm`)
    *   `dreb` (correlated with `reb`)

### Feature Engineering:

1.  **Points Per Minute (PPM)**: A new feature `'PPM'` was created by dividing a player's total `pts` by their `min` played. This normalizes scoring output by playing time, providing a per-minute scoring efficiency metric.
    *   Formula: `PPM = pts / min` (handling division by zero by setting to 0 if `min` is 0).
2.  **Efficiency Rating (EFF)**: A simplified Player Efficiency Rating `'EFF'` was engineered to encapsulate a player's overall impact. This metric combines positive contributions (points, rebounds, assists, steals, blocks) and subtracts negative contributions (missed field goals, missed free throws, turnovers), all normalized by games played.
    *   Formula: `EFF = (PTS + REB + AST + STL + BLK - ((FGA - FGM) + (FTA - FTM) + TOV)) / GP` (handling division by zero by setting to 0 if `gp` is 0).

### Data Cleaning:

1.  **Handling Null Values**: The dataset was inspected for missing values in performance columns. No null values were found, indicating a clean dataset in this regard. If nulls were present, a mean imputation strategy would have been applied for numerical columns.